In [1]:
import sys, torch, transformers, json
from pathlib import Path

print("python:", sys.version)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())

d:\VKB_Projects\Viettel_AI_Race\VAR_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
torch: 2.6.0+cu124
transformers: 5.13.1
cuda: True
device_count: 1


In [2]:
from huggingface_hub import list_repo_files

BASE_TOKENIZER = "vinai/phobert-base-v2"
CKPT_REPO = "leduckhai/VietMed-NER"
CKPT_SUBFOLDER = "phobert-base-v2-VietMed-NER"

repo_files = [p for p in list_repo_files(CKPT_REPO) if p.startswith(f"{CKPT_SUBFOLDER}/")]
print("ckpt_repo:", CKPT_REPO)
print("ckpt_subfolder:", CKPT_SUBFOLDER)
print("files:", repo_files)

ckpt_repo: leduckhai/VietMed-NER
ckpt_subfolder: phobert-base-v2-VietMed-NER
files: ['phobert-base-v2-VietMed-NER/.gitattributes', 'phobert-base-v2-VietMed-NER/config.json', 'phobert-base-v2-VietMed-NER/pytorch_model.bin']


In [3]:
from huggingface_hub import hf_hub_download

config_path = Path(
    hf_hub_download(
        repo_id=CKPT_REPO,
        filename="config.json",
        subfolder=CKPT_SUBFOLDER,
    )
)

print("downloaded_config:", config_path)

with open(config_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

print("model_type:", cfg.get("model_type"))
print("architectures:", cfg.get("architectures"))
print("num_labels:", cfg.get("num_labels"))
print("id2label:", cfg.get("id2label"))
print("label2id:", cfg.get("label2id"))

downloaded_config: C:\Users\VTN\.cache\huggingface\hub\models--leduckhai--VietMed-NER\snapshots\cccffb7de14423114f7d4bafc9f736b9d866e446\phobert-base-v2-VietMed-NER\config.json
model_type: roberta
architectures: ['RobertaForTokenClassification']
num_labels: None
id2label: {'0': 'I-ORGAN', '1': 'B-PREVENTIVEMED', '2': 'B-DISEASESYMTOM', '3': 'B-FOODDRINK', '4': 'B-ORGANIZATION', '5': 'B-OCCUPATION', '6': 'B-DRUGCHEMICAL', '7': 'I-FOODDRINK', '8': 'I-DISEASESYMTOM', '9': 'I-UNITCALIBRATOR', '10': 'I-DATETIME', '11': 'I-DIAGNOSTICS', '12': 'B-TRANSPORTATION', '13': 'B-GENDER', '14': 'B-AGE', '15': 'B-DATETIME', '16': 'B-LOCATION', '17': 'B-TREATMENT', '18': 'I-DRUGCHEMICAL', '19': 'I-PREVENTIVEMED', '20': 'I-TREATMENT', '21': 'I-ORGANIZATION', '22': '0', '23': 'B-UNITCALIBRATOR', '24': 'B-MEDDEVICETECHNIQUE', '25': 'I-OCCUPATION', '26': 'B-PERSONALCARE', '27': 'I-PERSONALCARE', '28': 'I-GENDER', '29': 'I-SURGERY', '30': 'I-AGE', '31': 'I-LOCATION', '32': 'B-DIAGNOSTICS', '33': 'I-MEDDEVIC

In [4]:
id2label = cfg.get("id2label", {})
labels = [id2label[k] for k in sorted(id2label, key=lambda x: int(x) if str(x).isdigit() else str(x))]
print("labels:", labels)

generic = all(str(x).startswith("LABEL_") for x in labels)
print("generic_labels:", generic)

labels: ['I-ORGAN', 'B-PREVENTIVEMED', 'B-DISEASESYMTOM', 'B-FOODDRINK', 'B-ORGANIZATION', 'B-OCCUPATION', 'B-DRUGCHEMICAL', 'I-FOODDRINK', 'I-DISEASESYMTOM', 'I-UNITCALIBRATOR', 'I-DATETIME', 'I-DIAGNOSTICS', 'B-TRANSPORTATION', 'B-GENDER', 'B-AGE', 'B-DATETIME', 'B-LOCATION', 'B-TREATMENT', 'I-DRUGCHEMICAL', 'I-PREVENTIVEMED', 'I-TREATMENT', 'I-ORGANIZATION', '0', 'B-UNITCALIBRATOR', 'B-MEDDEVICETECHNIQUE', 'I-OCCUPATION', 'B-PERSONALCARE', 'I-PERSONALCARE', 'I-GENDER', 'I-SURGERY', 'I-AGE', 'I-LOCATION', 'B-DIAGNOSTICS', 'I-MEDDEVICETECHNIQUE', 'B-ORGAN', 'B-SURGERY', 'I-TRANSPORTATION']
generic_labels: False


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER, use_fast=True)
print(type(tokenizer))
print(tokenizer.__class__.__name__)

d:\VKB_Projects\Viettel_AI_Race\VAR_env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VTN\.cache\huggingface\hub\models--vinai--phobert-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


<class 'transformers.models.phobert.tokenization_phobert.PhobertTokenizer'>
PhobertTokenizer


In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    CKPT_REPO,
    subfolder=CKPT_SUBFOLDER,
)
print(type(model))
print(model.config.id2label)

d:\VKB_Projects\Viettel_AI_Race\VAR_env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VTN\.cache\huggingface\hub\models--leduckhai--VietMed-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15734.17it/s]

<class 'transformers.models.roberta.modeling_roberta.RobertaForTokenClassification'>
{0: 'I-ORGAN', 1: 'B-PREVENTIVEMED', 2: 'B-DISEASESYMTOM', 3: 'B-FOODDRINK', 4: 'B-ORGANIZATION', 5: 'B-OCCUPATION', 6: 'B-DRUGCHEMICAL', 7: 'I-FOODDRINK', 8: 'I-DISEASESYMTOM', 9: 'I-UNITCALIBRATOR', 10: 'I-DATETIME', 11: 'I-DIAGNOSTICS', 12: 'B-TRANSPORTATION', 13: 'B-GENDER', 14: 'B-AGE', 15: 'B-DATETIME', 16: 'B-LOCATION', 17: 'B-TREATMENT', 18: 'I-DRUGCHEMICAL', 19: 'I-PREVENTIVEMED', 20: 'I-TREATMENT', 21: 'I-ORGANIZATION', 22: '0', 23: 'B-UNITCALIBRATOR', 24: 'B-MEDDEVICETECHNIQUE', 25: 'I-OCCUPATION', 26: 'B-PERSONALCARE', 27: 'I-PERSONALCARE', 28: 'I-GENDER', 29: 'I-SURGERY', 30: 'I-AGE', 31: 'I-LOCATION', 32: 'B-DIAGNOSTICS', 33: 'I-MEDDEVICETECHNIQUE', 34: 'B-ORGAN', 35: 'B-SURGERY', 36: 'I-TRANSPORTATION'}


In [7]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

text = "Bệnh nhân dùng doxycycline điều trị viêm tuyến mồ hôi và làm xét nghiệm CRP."
preds = ner(text)
preds

[{'entity_group': 'OCCUPATION',
  'score': np.float32(0.9934256),
  'word': 'Bệnh nhân',
  'start': None,
  'end': None},
 {'entity_group': '0',
  'score': np.float32(0.99942577),
  'word': 'dùng',
  'start': None,
  'end': None},
 {'entity_group': 'DRUGCHEMICAL',
  'score': np.float32(0.9939882),
  'word': 'doxycycline',
  'start': None,
  'end': None},
 {'entity_group': 'TREATMENT',
  'score': np.float32(0.994527),
  'word': 'điều trị',
  'start': None,
  'end': None},
 {'entity_group': 'DISEASESYMTOM',
  'score': np.float32(0.9939926),
  'word': 'viêm',
  'start': None,
  'end': None},
 {'entity_group': 'ORGAN',
  'score': np.float32(0.989831),
  'word': 'tuyến mồ hôi',
  'start': None,
  'end': None},
 {'entity_group': '0',
  'score': np.float32(0.9994284),
  'word': 'và làm',
  'start': None,
  'end': None},
 {'entity_group': 'DIAGNOSTICS',
  'score': np.float32(0.98853433),
  'word': 'xét nghiệm',
  'start': None,
  'end': None},
 {'entity_group': 'DIAGNOSTICS',
  'score': np.flo

In [ ]:
for p in preds:
    print(p)